# Toy example

In [1]:
from prefect import task, flow

@task
def hello():
    print("Hello Prefect")

@flow
def my_flow():
    hello()

my_flow()


18:06:23.521 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8218
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

18:06:27.655 | INFO    | Flow run 'purple-silkworm' - Beginning flow run 'purple-silkworm' for flow 'my-flow'

Hello Prefect


18:06:27.803 | INFO    | Task run 'hello-658' - Finished in state Completed()

18:06:28.813 | INFO    | Flow run 'purple-silkworm' - Finished in state Completed()

# ETL + Model

In [2]:
from prefect import task
import pandas as pd

from sklearn.datasets import fetch_california_housing

@task
def load_data():

    data = fetch_california_housing()

    X = pd.DataFrame(
        data.data,
        columns=data.feature_names
    )

    y = data.target

    print("Loaded data shape:", X.shape)

    return X, y

In [18]:
@task
def validate_data(X):

    assert X.isnull().sum().sum() == 0

    assert X.shape[0] > 1000

    print("Validation passed")

    return X

In [6]:
from sklearn.model_selection import train_test_split

@task
def split_data(X, y):

    X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )
    
    print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

    return X_train, X_test, y_train, y_test

In [15]:
import mlflow

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

mlflow.set_experiment(
    "house_pipeline"
)

@task
def train_model(
    X_train,
    X_test,
    y_train,
    y_test,
    n_estimators_list=[50, 100]
):
    
    best_mse = float("inf")
    best_model = None

    for n_estimators in n_estimators_list:

        model = RandomForestRegressor(
            n_estimators=n_estimators,
            random_state=42
        )

        with mlflow.start_run():

            model.fit(
                X_train,
                y_train
            )

            predictions = model.predict(
                X_test
            )

            mse = mean_squared_error(
                y_test,
                predictions
            )

            r2 = r2_score(
                y_test,
                predictions
            )

            mlflow.log_param(
                "n_estimators",
                n_estimators
            )

            mlflow.log_metric(
                        "r2",
                        r2      
            )

            mlflow.log_metric(
                "mse",
                mse
            )

            print(f"MSE: {mse}, R2: {r2}")

            if mse < best_mse:
                best_mse = mse
                best_model = model

    return best_model

In [14]:
from mlflow.models import infer_signature

@task
def save_best_model(model, X_train):

    signature = infer_signature(
        X_train,
        model.predict(X_train)
    )

    with mlflow.start_run():

        mlflow.sklearn.log_model(
            model,
            "best_model",
            signature=signature
        )

In [16]:
from prefect import flow

@flow
def training_pipeline(
    n_estimators_list=[50, 100, 200, 500]
):

    X, y = load_data()

    X = validate_data(X)

    X_train, X_test, y_train, y_test = split_data(
        X,
        y
    )

    model = train_model(
        X_train,
        X_test,
        y_train,
        y_test,
        n_estimators_list
    )

    save_best_model(
        model,
        X_train
    )

    return model


In [19]:
training_pipeline()

18:46:25.332 | INFO    | Flow run 'cooperative-tamarin' - Beginning flow run 'cooperative-tamarin' for flow 'training-pipeline'

Loaded data shape: (20640, 8)


18:46:25.343 | INFO    | Task run 'load_data-8e9' - Finished in state Completed()

Validation passed


18:46:25.352 | INFO    | Task run 'validate_data-75e' - Finished in state Completed()

...

18:47:30.182 | INFO    | Task run 'train_model-993' - Finished in state Completed()

2026/04/16 18:47:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 18:47:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


18:47:40.385 | INFO    | Task run 'save_best_model-b25' - Finished in state Completed()

18:47:40.475 | INFO    | Flow run 'cooperative-tamarin' - Finished in state Completed()

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [ ]:
training_pipeline(
    n_estimators=50
)

training_pipeline(
    n_estimators=200
)

In [11]:
for n in [50, 100, 200, 300]:

    training_pipeline(
        n_estimators=n
    )

18:32:29.331 | INFO    | Flow run 'organic-pogona' - Beginning flow run 'organic-pogona' for flow 'training-pipeline'

Loaded data shape: (20640, 8)


18:32:29.342 | INFO    | Task run 'load_data-9eb' - Finished in state Completed()

Validation passed


18:32:29.351 | INFO    | Task run 'validate_data-d23' - Finished in state Completed()

...

Train shape: (16512, 8), Test shape: (4128, 8)


18:32:58.990 | INFO    | Task run 'split_data-176' - Finished in state Completed()

MSE: 0.2534335583026607


18:33:21.899 | INFO    | Task run 'train_model-e69' - Finished in state Completed()

18:33:21.965 | INFO    | Flow run 'archetypal-marmoset' - Finished in state Completed()